# M7-B1 — Mesures d audit (à compléter)

## 1. Disparate impact du modèle, puis investigation

DI sur les étiquettes et sur les prédictions, puis investigation : étiquette confrontée à la durée réelle, erreurs et calibration par groupe contre les étiquettes `sejour_prolonge`.

In [1]:
from pathlib import Path

import joblib
import pandas as pd

DATA_PATH = Path("../data/dms_dataset.csv")
MODEL_PATH = Path("../legacy/dms_predictor_v1.joblib")
FEATURES = ["age", "nb_comorbidites", "imc", "sexe_bin"]

df = pd.read_csv(DATA_PATH)
model = joblib.load(MODEL_PATH)

df["sexe_bin"] = (df["sexe"] == "M").astype(int)
df["proba"] = model.predict_proba(df[FEATURES])[:, 1]
df["pred"] = (df["proba"] >= 0.5).astype(int)

print(df.shape)

(10000, 13)


### 1.1 Disparate impact : étiquettes puis prédictions

DI = taux de signalement « séjour prolongé » du groupe F divisé par celui du groupe M. Repère d'alerte conventionnel : 0,80. C'est un signal, pas un verdict.

In [2]:
def disparate_impact(rates: pd.Series, group_a: str = "F", group_b: str = "M") -> float:
    return rates[group_a] / rates[group_b]


rates = df.groupby("sexe")[["sejour_prolonge", "pred"]].mean()
print(rates.round(3))
print("DI F/M sur les étiquettes :", round(disparate_impact(rates["sejour_prolonge"]), 3))
print("DI F/M sur les prédictions :", round(disparate_impact(rates["pred"]), 3))

      sejour_prolonge   pred
sexe                        
F               0.321  0.141
M               0.491  0.486
DI F/M sur les étiquettes : 0.653
DI F/M sur les prédictions : 0.291


### 1.2 Investigation : la durée réelle explique-t-elle l'écart ?

Si les femmes étaient signalées moins souvent parce que leurs séjours sont plus courts, l'écart serait compréhensible.

In [3]:
print(df.groupby("sexe")["dms_jours"].describe().round(2))

       count  mean   std  min  25%  50%  75%   max
sexe                                              
F     5011.0  5.62  2.46  1.0  3.9  5.6  7.3  14.9
M     4989.0  5.59  2.45  1.0  3.8  5.5  7.3  15.3


### 1.3 Investigation : à durée réelle égale, l'étiquette est-elle attribuée de la même façon ?

Une durée longue ne fait pas à elle seule un séjour prolongé. On compare donc seulement la part d'étiquettes « prolongé » par tranche de durée réelle et par sexe, sans considérer la durée comme une vérité de référence.

In [4]:
DUREE_BINS = [0, 5.5, 7, 9, 16]

df["tranche_duree"] = pd.cut(df["dms_jours"], DUREE_BINS)

label_by_duration = df.pivot_table(
    index="tranche_duree",
    columns="sexe",
    values="sejour_prolonge",
    aggfunc="mean",
    observed=True,
)
print(label_by_duration.round(3))

print(df.groupby(["sexe", "sejour_prolonge"])["dms_jours"].agg(["min", "max"]).round(1))

sexe               F    M
tranche_duree            
(0.0, 5.5]     0.000  0.0
(5.5, 7.0]     0.634  1.0
(7.0, 9.0]     0.629  1.0
(9.0, 16.0]    0.658  1.0
                      min   max
sexe sejour_prolonge           
F    0                1.0  14.9
     1                5.6  14.3
M    0                1.0   5.5
     1                5.6  15.3


### 1.4 Erreurs par groupe : FNR et FPR du modèle contre les étiquettes

FNR : cas étiquetés « prolongé » que le modèle ne signale pas. FPR : cas étiquetés « standard » que le modèle signale. Ces taux mesurent la fidélité du modèle à ses étiquettes, pas leur justesse.

In [5]:
def error_rates(group: pd.DataFrame, reference: str = "sejour_prolonge", prediction: str = "pred") -> pd.Series:
    positives = group[group[reference] == 1]
    negatives = group[group[reference] == 0]
    return pd.Series(
        {
            "FNR": 1 - positives[prediction].mean(),
            "FPR": negatives[prediction].mean(),
        }
    )


errors = pd.DataFrame({sexe: error_rates(group) for sexe, group in df.groupby("sexe")}).T
print(errors.round(3))

     FNR    FPR
F  0.628  0.033
M  0.242  0.223


### 1.5 Calibration par groupe

Probabilité moyenne prédite comparée au taux d'étiquettes « prolongé », pour chaque sexe.

In [6]:
calibration = pd.DataFrame(
    {
        "proba_moyenne": df.groupby("sexe")["proba"].mean(),
        "taux_etiquette": df.groupby("sexe")["sejour_prolonge"].mean(),
    }
)
calibration["ecart"] = calibration["proba_moyenne"] - calibration["taux_etiquette"]
print(calibration.round(3))

      proba_moyenne  taux_etiquette  ecart
sexe                                      
F             0.321           0.321  0.000
M             0.491           0.491 -0.001


## 2. Ressources (psutil)

In [7]:
# TODO


## 3. Comparaison à 2 alternatives

In [8]:
# TODO
